# CS5483 Project 2 Group 1

In [ ]:
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import warnings
from pathlib import Path
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import davies_bouldin_score, mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
try:
    from IPython.display import display
except ImportError:
    display = print

In [ ]:
RANDOM_SEED = 42

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", DEVICE)

## 1. Data preprocessing

### Statistics for Precipitations

In [ ]:
df_x = pd.read_csv("data/x_train_final_asAbTs5.csv")

if "Precipitations" in df_x.columns:
    print("Descriptive statistics for Precipitations:")
    print(df_x["Precipitations"].describe())

    non_zero_count = (df_x["Precipitations"] != 0).sum()
    non_zero_ratio = (df_x["Precipitations"] != 0).mean()
    print("Number of non-zero values:", non_zero_count)
    print("Ratio of non-zero values:", non_zero_ratio)

    value_counts = df_x["Precipitations"].value_counts().sort_index()
    print("Value distribution (top 10):")
    print(value_counts.head(10))


    range_ratio = ((df_x["Precipitations"] >= 0) & (df_x["Precipitations"] <= 3)).mean()
    print("Ratio of values in range 0–3:", range_ratio)
else:
    print("Column 'Precipitations' does not exist in dataset.")

### Statistics for HauteurNeige

In [ ]:
df_x = pd.read_csv("data/x_train_final_asAbTs5.csv")

if "HauteurNeige" in df_x.columns:
    print("HauteurNeige Column statistics：")
    print(df_x["HauteurNeige"].describe())  
    print("Number of non-zero values：", (df_x["HauteurNeige"] != 0).sum())
    print("proportion of non-zero values：", (df_x["HauteurNeige"] != 0).mean())
else:
    print("Column HauteurNeige does not exist in dataset")

### Preprocessing

In [ ]:
df_x = pd.read_csv("data/x_train_final_asAbTs5.csv")
df_y = pd.read_csv("data/y_train_final_YYyFil7.csv")

invalid_ratio = df_y.iloc[:, 1]
df = pd.concat([df_x, invalid_ratio], axis=1)

df = df.drop(columns=["HauteurNeige"])
df['rain_flag'] = (df['Precipitations'] > 0).astype(int)

df.to_csv("data/XAndY_Preprocess.csv", index=False)

print(df.head())


### Source data visualization

In [ ]:
# 1. Load preprocessed dataset
df = pd.read_csv("data/XAndY_Preprocess.csv")

# 2. Histogram of invalid_ratio
plt.figure(figsize=(6,4))
plt.hist(df['invalid_ratio'], bins=30, color='steelblue')
plt.xlabel("Invalid Ratio")
plt.ylabel("Count")
plt.title("Distribution of Invalid Ratio")
plt.show()

# 3. Monthly violation pattern (line chart)
month_mean = df.groupby('month_of_year')['invalid_ratio'].mean()
plt.figure(figsize=(8,4))
plt.plot(month_mean.index, month_mean.values, marker='o', color='green')
plt.xlabel("Month")
plt.ylabel("Average Invalid Ratio")
plt.title("Monthly Violation Pattern")
plt.grid(True)
plt.show()

# 4. Heatmap: Day of Week × Hour
pivot = df.pivot_table(values='invalid_ratio', index='day_of_week', columns='hour', aggfunc='mean')
plt.figure(figsize=(12,5))
sns.heatmap(pivot, cmap='YlOrRd')
plt.title("Heatmap: Day of Week × Hour (Violation Rate)")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.show()


## 2. Clustering Analysis

### Load & Process Data

In [ ]:
df = pd.read_csv("data/XAndY_Preprocess.csv")
df = df.drop(["Unnamed: 0"], axis = 1)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

features = df[['longitude_scaled', 'latitude_scaled',
               'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']]
features_space = df[['longitude_scaled', 'latitude_scaled']]
features_time = df[['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']]

features_scaled = StandardScaler().fit_transform(features)
features_space_scaled = StandardScaler().fit_transform(features_space)
features_time_scaled = StandardScaler().fit_transform(features_time)

target = df['invalid_ratio']

### 2.1 K-Means Clustering

In [ ]:
RUN_FIND_K = False

In [ ]:
features_np = features_scaled.astype(np.float32)
features_np_sampled = resample(
    features_np,
    replace = False,
    n_samples = 40000,
    random_state = RANDOM_SEED
)

features_space_np = features_space_scaled.astype(np.float32)
features_space_np_sampled = resample(
    features_space_np,
    replace = False,
    n_samples = 40000,
    random_state = RANDOM_SEED
)

features_time_np = features_time_scaled.astype(np.float32)
features_time_np_sampled = resample(
    features_time_np,
    replace = False,
    n_samples = 40000,
    random_state = RANDOM_SEED
)

#### 2.1.1 Using Both Space And Time Features

In [ ]:
results = []

if RUN_FIND_K:
    for k in tqdm(range(2, 11)):
        kmeans = faiss.Kmeans(
            d = features_np.shape[1],
            k = k,
            niter = 20,
            gpu = True,
            verbose = False
        )
        kmeans.train(features_np)
        
        _, labels_sampled = kmeans.index.search(features_np_sampled, 1)
        s = silhouette_score(features_np_sampled, labels_sampled.flatten())
        results.append((k, s))

else:
    # This is an output from previous run.
    results = [
        (2, 0.21230898797512054),
        (3, 0.2332783192396164),
        (4, 0.2509840726852417),
        (5, 0.2880259156227112),
        (6, 0.29532095789909363),
        (7, 0.29926028847694397),
        (8, 0.2888481020927429),
        (9, 0.2521010637283325),
        (10, 0.2639625370502472)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means (ST): Silhouette Score vs Number of Clusters")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters = 7, random_state = RANDOM_SEED, n_init = 10)
labels_kmeans_st = kmeans.fit_predict(features_scaled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_scaled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_kmeans_st, s = 5)
plt.title("K-Means Cluster Assignment (ST)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.1.2 Using Only Space Features

In [ ]:
results = []

if RUN_FIND_K:
    for k in tqdm(range(2, 11)):
        kmeans = faiss.Kmeans(
            d = features_space_np.shape[1],
            k = k,
            niter = 20,
            gpu = True,
            verbose = False
        )
        kmeans.train(features_space_np)
        
        _, labels_sampled = kmeans.index.search(features_space_np_sampled, 1)
        s = silhouette_score(features_space_np_sampled, labels_sampled.flatten())
        results.append((k, s))

else:
    # This is an output from previous run.
    results = [
        (2, 0.590848445892334),
        (3, 0.5952897667884827),
        (4, 0.5494586229324341),
        (5, 0.5342718958854675),
        (6, 0.5057369470596313),
        (7, 0.47282958030700684),
        (8, 0.44575735926628113),
        (9, 0.4280731976032257),
        (10, 0.4162769913673401)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means (S): Silhouette Score vs Number of Clusters")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters = 3, random_state = RANDOM_SEED, n_init = 10)
labels_kmeans_s = kmeans.fit_predict(features_space_scaled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_space_scaled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_kmeans_s, s = 5)
plt.title("K-Means Cluster Assignment (S)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.1.3 Using Only Time Features

In [ ]:
results = []

if RUN_FIND_K:
    for k in tqdm(range(2, 11)):
        kmeans = faiss.Kmeans(
            d = features_time_np.shape[1],
            k = k,
            niter = 20,
            gpu = True,
            verbose = False
        )
        kmeans.train(features_time_np)
        
        _, labels_sampled = kmeans.index.search(features_time_np_sampled, 1)
        s = silhouette_score(features_time_np_sampled, labels_sampled.flatten())
        results.append((k, s))

else:
    # This is an output from previous run.
    results = [
        (2, 0.20377245545387268),
        (3, 0.256518691778183),
        (4, 0.2844865620136261),
        (5, 0.32636430859565735),
        (6, 0.327036052942276),
        (7, 0.34803441166877747),
        (8, 0.32171016931533813),
        (9, 0.309508740901947),
        (10, 0.3284476697444916)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means (T): Silhouette Score vs Number of Clusters")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters = 7, random_state = RANDOM_SEED, n_init = 10)
labels_kmeans_t = kmeans.fit_predict(features_time_scaled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_time_scaled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_kmeans_t, s = 5)
plt.title("K-Means Cluster Assignment (T)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.1.4 Check

In [ ]:
dbi_kmeans_st = davies_bouldin_score(features_scaled, labels_kmeans_st)
dbi_kmeans_s = davies_bouldin_score(features_space_scaled, labels_kmeans_s)
dbi_kmeans_t = davies_bouldin_score(features_time_scaled, labels_kmeans_t)

print(f"K-Means DBI (ST): {dbi_kmeans_st:.4f}")
print(f"K-Means DBI (S):  {dbi_kmeans_s:.4f}")
print(f"K-Means DBI (T):  {dbi_kmeans_t:.4f}")

In [ ]:
def r2_clustering(y, labels):
    ss_total = np.sum((y - y.mean()) ** 2)
    ss_within = sum(
        np.sum((y[labels == c] - y[labels == c].mean()) ** 2)
        for c in np.unique(labels)
    )
    return 1 - ss_within / ss_total

target_np = target.to_numpy()
r2_kmeans_st = r2_clustering(target_np, labels_kmeans_st)
r2_kmeans_s = r2_clustering(target_np, labels_kmeans_s)
r2_kmeans_t = r2_clustering(target_np, labels_kmeans_t)

print(f"K-Means R² (ST): {r2_kmeans_st:.4f}")
print(f"K-Means R² (S):  {r2_kmeans_s:.4f}")
print(f"K-Means R² (T):  {r2_kmeans_t:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (18, 5))

for ax, labels, title in zip(
    axes,
    [labels_kmeans_st, labels_kmeans_s, labels_kmeans_t],
    ["K-Means (ST)", "K-Means (S)", "K-Means (T)"]
):
    clusters = np.unique(labels)
    data = [target_np[labels == c] for c in clusters]
    ax.boxplot(data, labels = [str(c) for c in clusters])
    ax.set_title(f"{title}: Invalid Ratio by Cluster")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Invalid Ratio")

plt.tight_layout()
plt.show()

### 2.2 DBSCAN

In [ ]:
RUN_FIND_EPS = False

In [ ]:
idx = np.random.RandomState(RANDOM_SEED).choice(
    len(features_scaled),
    size = 40000,
    replace = False
)
features_sampled = features_scaled[idx].astype(np.float32)
target_sampled = target.to_numpy()[idx]

idx = np.random.RandomState(RANDOM_SEED).choice(
    len(features_space_scaled),
    size = 40000,
    replace = False
)
features_space_sampled = features_space_scaled[idx].astype(np.float32)
target_space_sampled = target.to_numpy()[idx]

idx = np.random.RandomState(RANDOM_SEED).choice(
    len(features_time_scaled),
    size = 40000,
    replace = False
)
features_time_sampled = features_time_scaled[idx].astype(np.float32)
target_time_sampled = target.to_numpy()[idx]

#### 2.2.1 Using Both Space And Time Features

In [ ]:
results = []

if RUN_FIND_EPS:
    for eps in tqdm([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]):
        labels = DBSCAN(
            eps = eps,
            min_samples = 100,
            algorithm = "ball_tree",
            n_jobs = -1
        ).fit_predict(features_sampled)
    
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        results.append((eps, n_clusters, n_noise))

else:
    # This is an output from previous run.
    results = [
        (0.2, 69, 2495),
        (0.3, 70, 939),
        (0.4, 72, 141),
        (0.5, 60, 34),
        (0.6, 48, 26),
        (0.7, 36, 26),
        (0.8, 24, 26),
        (0.9, 12, 26),
        (1.0, 6, 2),
        (1.1, 6, 2),
        (1.2, 5, 2),
        (1.3, 3, 2),
        (1.4, 1, 2),
        (1.5, 1, 2)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.title("DBSCAN (ST): Number of Clusters vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Clusters")
plt.grid(True)
plt.subplot(122)
plt.plot([r[0] for r in results], [r[2] for r in results], marker = 'o')
plt.title("DBSCAN (ST): Noise Points vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Noise Points")
plt.grid(True)
plt.tight_layout()
plt.show()

* Memory constraints prevent running predictions on the full dataset, so visualization is performed using a sampled subset.

In [ ]:
dbscan = DBSCAN(eps = 0.5, min_samples = 100, algorithm = "ball_tree", n_jobs = -1)
labels_dbscan_st = dbscan.fit_predict(features_sampled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_sampled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_dbscan_st, s = 5)
plt.title("DBSCAN Cluster Assignment (ST)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target_sampled, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.2.2 Using Only Space Features

In [ ]:
results = []

if RUN_FIND_EPS:
    for eps in tqdm([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]):
        labels = DBSCAN(
            eps = eps,
            min_samples = 100,
            algorithm = "ball_tree",
            n_jobs = -1
        ).fit_predict(features_space_sampled)
    
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        results.append((eps, n_clusters, n_noise))

else:
    # This is an output from previous run.
    results = [
        (0.2, 1, 3),
        (0.3, 1, 2),
        (0.4, 1, 2),
        (0.5, 1, 2),
        (0.6, 1, 2),
        (0.7, 1, 2),
        (0.8, 1, 2),
        (0.9, 1, 2),
        (1.0, 1, 2),
        (1.1, 1, 2),
        (1.2, 1, 2),
        (1.3, 1, 2),
        (1.4, 1, 2),
        (1.5, 1, 2)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.title("DBSCAN (S): Number of Clusters vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Clusters")
plt.grid(True)
plt.subplot(122)
plt.plot([r[0] for r in results], [r[2] for r in results], marker = 'o')
plt.title("DBSCAN (S): Noise Points vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Noise Points")
plt.grid(True)
plt.tight_layout()
plt.show()

* Memory constraints prevent running predictions on the full dataset, so visualization is performed using a sampled subset.

In [ ]:
dbscan = DBSCAN(eps = 0.3, min_samples = 100, algorithm = "ball_tree", n_jobs = -1)
labels_dbscan_s = dbscan.fit_predict(features_space_sampled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_space_sampled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_dbscan_s, s = 5)
plt.title("DBSCAN Cluster Assignment (S)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target_space_sampled, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.2.3 Using Only Time Features

In [ ]:
results = []

if RUN_FIND_EPS:
    for eps in tqdm([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]):
        labels = DBSCAN(
            eps = eps,
            min_samples = 100,
            algorithm = "ball_tree",
            n_jobs = -1
        ).fit_predict(features_time_sampled)
    
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        results.append((eps, n_clusters, n_noise))

else:
    # This is an output from previous run.
    results = [
        (0.2, 72, 24),
        (0.3, 72, 24),
        (0.4, 72, 24),
        (0.5, 60, 24),
        (0.6, 48, 24),
        (0.7, 36, 24),
        (0.8, 24, 24),
        (0.9, 12, 24),
        (1.0, 6, 0),
        (1.1, 6, 0),
        (1.2, 5, 0),
        (1.3, 3, 0),
        (1.4, 1, 0),
        (1.5, 1, 0)
    ]

print(results)

In [ ]:
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.plot([r[0] for r in results], [r[1] for r in results], marker = 'o')
plt.title("DBSCAN (T): Number of Clusters vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Clusters")
plt.grid(True)
plt.subplot(122)
plt.plot([r[0] for r in results], [r[2] for r in results], marker = 'o')
plt.title("DBSCAN (T): Noise Points vs EPS")
plt.xlabel("EPS")
plt.ylabel("Number of Noise Points")
plt.grid(True)
plt.tight_layout()
plt.show()

* Memory constraints prevent running predictions on the full dataset, so visualization is performed using a sampled subset.

In [ ]:
dbscan = DBSCAN(eps = 0.5, min_samples = 100, algorithm = "ball_tree", n_jobs = -1)
labels_dbscan_t = dbscan.fit_predict(features_time_sampled)

In [ ]:
pca = PCA(n_components = 2, random_state = RANDOM_SEED)
X_pca = pca.fit_transform(features_time_sampled)
plt.figure(figsize = (10, 5))
plt.subplot(121)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = labels_dbscan_t, s = 5)
plt.title("DBSCAN Cluster Assignment (T)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.subplot(122)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c = target_time_sampled, s = 5)
plt.title("Invalid Ratio Distribution")
plt.xlabel("PC1")
plt.tight_layout()
plt.show()

#### 2.2.4 Check

In [ ]:
dbscan_configs = [
    ("DBSCAN (ST)", features_sampled, labels_dbscan_st),
    ("DBSCAN (S)",  features_space_sampled, labels_dbscan_s),
    ("DBSCAN (T)",  features_time_sampled, labels_dbscan_t),
]

for name, X, labels in dbscan_configs:
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    if n_clusters < 2:
        print(f"{name} DBI: N/A (only {n_clusters} cluster)")
    else:
        dbi = davies_bouldin_score(X[mask], labels[mask])
        print(f"{name} DBI: {dbi:.4f}")

In [ ]:
dbscan_target_configs = [
    ("DBSCAN (ST)", labels_dbscan_st, target_sampled),
    ("DBSCAN (S)",  labels_dbscan_s, target_space_sampled),
    ("DBSCAN (T)",  labels_dbscan_t, target_time_sampled),
]

for name, labels, y in dbscan_target_configs:
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    if n_clusters < 2:
        print(f"{name} R²: N/A (only {n_clusters} cluster)")
    else:
        r2 = r2_clustering(y[mask], labels[mask])
        print(f"{name} R²: {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (18, 5))

for ax, (name, labels, y) in zip(axes, dbscan_target_configs):
    mask = labels != -1
    clusters = np.unique(labels[mask])
    if len(clusters) < 2:
        ax.text(0.5, 0.5, "Only 1 cluster", ha = "center", va = "center",
                transform = ax.transAxes, fontsize = 14)
    else:
        data = [y[labels == c] for c in clusters]
        ax.boxplot(data, labels = [str(c) for c in clusters])
    ax.set_title(f"{name}: Invalid Ratio by Cluster")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Invalid Ratio")

plt.tight_layout()
plt.show()

### Save Results

#### K-MEANS S

In [ ]:
df_tmp = df.copy()
df_tmp["cluster"] = labels_kmeans_s
df_tmp.to_csv("data/with_cluster_sampled.csv", index = False)

## 3. Regression Analysis

### Load & Process Data

In this section, we load the preprocessed dataset and prepare features for regression analysis. 

The target variable is **invalid_ratio**, and we construct input features based on spatial and temporal information. 
All features are standardized before model training to ensure comparability across different scales.

In [ ]:
# Load data
df = pd.read_csv("data/XAndY_Preprocess.csv")

# Drop unnecessary columns (if exist)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Feature engineering (time encoding)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Feature selection
features = df[['longitude_scaled', 'latitude_scaled',
               'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']]

target = df['invalid_ratio']

# Standardization
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

### 3.1 Linear Regression

We first apply a Linear Regression model as a baseline to predict the target variable **invalid_ratio**.

Linear Regression assumes a linear relationship between input features and the target. Although it may not capture complex nonlinear patterns, it provides a useful reference for evaluating more advanced models.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled, target, test_size=0.2, random_state=42
)

# Train model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Prediction
y_pred_lr = lr.predict(X_test)

In [ ]:
# Evaluation
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression RMSE:", rmse_lr)
print("Linear Regression R2:", r2_lr)

In [ ]:
plt.scatter(y_test, y_pred_lr, alpha=0.3)
plt.xlabel("True Values")
plt.ylabel("Predictions")
plt.title("Linear Regression: True vs Predicted")
plt.show()

The Linear Regression model provides a baseline performance. 

Due to its simplicity, it may not fully capture complex spatial-temporal relationships in the data, which can lead to limited predictive accuracy.

### 3.2 Random Forest

We further apply a Random Forest Regressor to capture nonlinear relationships in the data.

Random Forest is an ensemble learning method that builds multiple decision trees and aggregates their predictions. It is capable of modeling complex interactions between spatial and temporal features, making it more flexible than linear models.

In [ ]:
# Train model
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Prediction
y_pred_rf = rf.predict(X_test)

In [ ]:
# Evaluation
rmse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R2:", r2_rf)

In [ ]:
feature_importance = pd.Series(rf.feature_importances_, index=features.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print(feature_importance)

In [ ]:
feature_importance.plot(kind='bar')
plt.title("Feature Importance (Random Forest)")
plt.show()

The Random Forest model shows improved performance compared to Linear Regression, indicating that nonlinear patterns exist in the data.

Feature importance analysis suggests that both spatial and temporal variables contribute to predicting the target, highlighting the value of combining these features.

### 3.3 Model Comparison

The Random Forest model shows improved performance compared to Linear Regression, indicating that nonlinear patterns exist in the data.

Feature importance analysis suggests that both spatial and temporal variables contribute to predicting the target, highlighting the value of combining these features.

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "RMSE": [rmse_lr, rmse_rf],
    "R2": [r2_lr, r2_rf]
})

results

In [ ]:
results.set_index("Model")[["RMSE", "R2"]].plot(kind="bar", figsize=(8,5))
plt.title("Model Comparison")
plt.xticks(rotation=0)
plt.show()

The comparison shows that the Random Forest model outperforms Linear Regression in terms of both RMSE and R².

This suggests that the relationship between the input features and the target variable is not purely linear. The Random Forest model is better able to capture complex patterns and interactions in the data.

Overall, incorporating nonlinear models leads to improved predictive performance.

#### 3.3.1 Effect of Clustering Features

We further investigate the impact of incorporating clustering-based features into the regression models.

Specifically, we compare model performance with and without the inclusion of region cluster labels derived from the clustering analysis.

In [ ]:
df = pd.read_csv("data/with_cluster_sampled.csv")

features = df[
    [
        'longitude_scaled',
        'latitude_scaled',
        'hour_sin',
        'hour_cos',
        'dow_sin',
        'dow_cos',
        'cluster'   # ← New feature
    ]
]

In [ ]:
X = features
y = df["invalid_ratio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")
    
y_pred = rf.predict(X_test)
evaluate(y_test, y_pred, "RF + Cluster")

| Model          | RMSE   | R²     |
|----------------|--------|--------|
| RF baseline    | 0.0839 | 0.3822 |
| RF + cluster   | 0.3255 | 0.2066 |

Adding cluster labels does not improve prediction performance.

#### 3.3.2 Feature Importance

In [ ]:
importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp

The feature importance results show that spatial features (latitude and longitude) dominate the prediction, indicating that location is the primary factor influencing parking violations. Temporal features (hour and day patterns) have moderate impact, while the cluster feature contributes relatively little. This suggests that the original spatial variables already capture most of the regional information, making the cluster feature largely redundant.

## 4. Evaluation and Visualization

* Uniformly set drawing styles, import core libraries, define file paths and parameter constants, and check whether data files exist

In [ ]:
%matplotlib inline

sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (9, 5)

ROOT = Path(".").resolve()
DATA = ROOT / "data" / "XAndY_Preprocess.csv"
DATA_CLUSTER = ROOT / "data" / "with_cluster_sampled.csv"
FIG = ROOT / "figures_phase3"
FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SAMPLE = 120_000
TEST_SIZE = 0.22
RF_TREES = 72
BOOT_B = 150
PERM_REPEAT = 8
PERM_MAX_ROWS = 9000

print("data:", DATA.exists(), DATA)
print("Cluster label file:", DATA_CLUSTER.exists(), DATA_CLUSTER)

* Load data, clean samples, split into training and test sets, train a random forest regression model to predict invalid_ratio, and output the model's RMSE and Spearman correlation coefficient evaluation results.

In [ ]:
assert DATA.exists(), "请将 XAndY_Preprocess.csv 放在 data/ 目录"

raw = pd.read_csv(DATA).drop(columns=["Unnamed: 0"], errors="ignore")
if N_SAMPLE is not None and len(raw) > N_SAMPLE:
    raw = raw.sample(n=N_SAMPLE, random_state=RANDOM_STATE).reset_index(drop=True)

y_all = raw["invalid_ratio"]
X_all = raw.drop(columns=["invalid_ratio"])
X_all = X_all.apply(pd.to_numeric, errors="coerce")
y_all = pd.to_numeric(y_all, errors="coerce")
_tab = pd.concat([X_all, y_all.rename("__y__")], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
X_all = _tab.drop(columns=["__y__"])
y_all = _tab["__y__"]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

rf = RandomForestRegressor(
    n_estimators=RF_TREES,
    max_depth=18,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
rho, _ = stats.spearmanr(y_test, y_pred)
print(f"RMSE={rmse:.5f}  Spearman={rho:.4f}  n_test={len(y_test):,}")

* Calculate the 95% confidence interval of the test set RMSE using Bootstrap resampling, plot the distribution histogram, and save the image along with the metric results

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = np.arange(len(y_test))
boot = []
for _ in range(BOOT_B):
    s = rng.choice(idx, size=len(idx), replace=True)
    e = y_test.values[s] - y_pred[s]
    boot.append(float(np.sqrt(np.mean(e ** 2))))
boot = np.array(boot)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Bootstrap RMSE: 均值={boot.mean():.5f}  95% CI [{lo:.5f}, {hi:.5f}]")

fig, ax = plt.subplots()
ax.hist(boot, bins=28, color="steelblue", edgecolor="white", alpha=0.9)
ax.axvline(rmse, color="crimson", lw=2, label=f"point RMSE={rmse:.4f}")
ax.axvline(lo, color="gray", ls="--")
ax.axvline(hi, color="gray", ls="--", label="95% CI")
ax.set_title("Bootstrap distribution of test RMSE (resampled rows)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "01_bootstrap_rmse.png", bbox_inches="tight")
plt.show()

pd.DataFrame([{"rmse_point": rmse, "boot_mean": boot.mean(), "ci95_low": lo, "ci95_high": hi}]).to_csv(
    FIG / "metrics_bootstrap_rmse.csv", index=False
)

* Perform permutation feature importance analysis on a subset of the test set, calculating the impact on RMSE after each feature is shuffled

In [ ]:
n_perm = min(PERM_MAX_ROWS, len(X_test))
perm_idx = np.random.default_rng(RANDOM_STATE).choice(len(X_test), size=n_perm, replace=False)
X_sub = X_test.iloc[perm_idx]
y_sub = y_test.iloc[perm_idx]

r = permutation_importance(
    rf,
    X_sub,
    y_sub,
    n_repeats=PERM_REPEAT,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
imp_mean = pd.Series(r.importances_mean, index=X_all.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
imp_mean.tail(12).plot(kind="barh", ax=ax, color="darkslategray")
ax.set_title("Permutation importance (↑ = RMSE worsens when column shuffled)")
ax.set_xlabel("mean RMSE increase")
fig.tight_layout()
fig.savefig(FIG / "02_permutation_importance.png", bbox_inches="tight")
plt.show()

imp_mean.to_frame("mean_rmse_increase").to_csv(FIG / "metrics_permutation_importance.csv")

* Plot the kernel density distribution of actual values and predicted values to compare distribution differences, and then calculate and plot a reliability calibration chart through binning to comprehensively assess the consistency and accuracy of the model's prediction results.

In [ ]:
yt = y_test.values.astype(float)
yp = y_pred.astype(float)

fig, ax = plt.subplots()
sns.kdeplot(yt, ax=ax, fill=True, alpha=0.35, label="y true", color="tab:blue")
sns.kdeplot(yp, ax=ax, fill=True, alpha=0.35, label="y pred", color="tab:orange")
ax.set_xlim(0, 1)
ax.set_title("Density: true vs predicted invalid_ratio (test)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "03_density_true_vs_pred.png", bbox_inches="tight")
plt.show()

n_bins = 12
order = np.argsort(yp)
yp_s, yt_s = yp[order], yt[order]
parts = np.array_split(np.arange(len(yp_s)), n_bins)
centers_p, centers_t = [], []
for p in parts:
    if len(p) == 0:
        continue
    centers_p.append(yp_s[p].mean())
    centers_t.append(yt_s[p].mean())
fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.plot(centers_p, centers_t, "o-", ms=6, color="darkgreen")
ax.set_xlabel("mean predicted (bin)")
ax.set_ylabel("mean true (bin)")
ax.set_title("Reliability / calibration (test)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIG / "04_reliability_calibration.png", bbox_inches="tight")
plt.show()

* Sort the test set from high to low according to the model's predicted values, calculate and plot the cumulative actual mean curve, intuitively showing the model's ability to identify samples with a high invalid rate and the effect of concentration aggregation.

In [ ]:
res = yp - yt
tc = X_test["total_count"].values if "total_count" in X_test.columns else np.ones(len(yt))

order = np.argsort(-yp)
cum_y = np.cumsum(yt[order]) / np.arange(1, len(yt) + 1)
frac = np.arange(1, len(yt) + 1) / len(yt)
global_mean = yt.mean()
fig, ax = plt.subplots()
ax.plot(frac, cum_y, color="navy", lw=1.8, label="cum mean y | sorted by pred desc")
ax.axhline(global_mean, color="gray", ls="--", label=f"overall mean={global_mean:.3f}")
ax.set_xlabel("fraction of test population (top predictions first)")
ax.set_ylabel("cumulative mean true invalid_ratio")
ax.set_title("Regression-style cumulative gain / concentration curve")
ax.legend(loc="upper right")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(FIG / "06_cumulative_gain_regression.png", bbox_inches="tight")
plt.show()

* Standardized based on spatiotemporal features such as latitude, longitude, and time, first using K-Means to try different numbers of clusters, then using DBSCAN to traverse the eps parameter to evaluate clustering quality, and finally plotting cluster effect charts and generating a heatmap of average invalid rates by cluster × month, fully completing spatiotemporal feature clustering analysis and visualization.

In [ ]:
feat_st = ["longitude_scaled", "latitude_scaled", "hour", "day_of_week", "month_of_year"]
Z = StandardScaler().fit_transform(X_all[feat_st].values.astype(float))
n_sub = min(25000, Z.shape[0])
rs = np.random.default_rng(RANDOM_STATE)
ix = rs.choice(Z.shape[0], size=n_sub, replace=False)
Z_sub = Z[ix]

ks = [3, 5, 7, 9, 11]
db_scores = []
for k in ks:
    lab = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(Z_sub)
    db_scores.append(davies_bouldin_score(Z_sub, lab))

fig, ax = plt.subplots()
ax.bar([str(k) for k in ks], db_scores, color="teal", edgecolor="white")
ax.set_xlabel("k (KMeans on scaled ST subset)")
ax.set_ylabel("Davies–Bouldin index (lower is better)")
ax.set_title("Clustering quality snapshot (NOT full silhouette sweep)")
fig.tight_layout()
fig.savefig(FIG / "07_davies_bouldin_bar.png", bbox_inches="tight")
plt.show()

eps_grid = [round(x, 1) for x in np.arange(0.2, 1.55, 0.1)]
DB_MIN_SAMPLES = 100
db_rows = []
for eps in eps_grid:
    lab = DBSCAN(
        eps=eps,
        min_samples=DB_MIN_SAMPLES,
        algorithm="ball_tree",
        n_jobs=-1,
    ).fit_predict(Z_sub)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = int((lab == -1).sum())
    noise_frac = n_noise / max(len(lab), 1)
    core_mask = lab >= 0
    db_eps = np.nan
    if n_clusters >= 2 and core_mask.sum() > 1:
        try:
            db_eps = davies_bouldin_score(Z_sub[core_mask], lab[core_mask])
        except ValueError:
            db_eps = np.nan
    db_rows.append(
        {
            "eps": eps,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "noise_frac": noise_frac,
            "davies_bouldin_nonnoise": db_eps,
        }
    )
db_df = pd.DataFrame(db_rows)
display(db_df.head(10))
db_df.to_csv(FIG / "metrics_dbscan_eps_sweep.csv", index=False)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4))
ax_a.plot(db_df["eps"], db_df["n_clusters"], "o-", color="darkviolet", lw=1.5)
ax_a.set_xlabel("eps (same scale as standardized Z_sub)")
ax_a.set_ylabel("n_clusters (noise label -1 excluded)")
ax_a.set_title("DBSCAN: cluster count vs eps")

ax_b.plot(db_df["eps"], db_df["noise_frac"], "s-", color="coral", lw=1.5)
ax_b.set_xlabel("eps")
ax_b.set_ylabel("noise fraction")
ax_b.set_title("DBSCAN: noise fraction vs eps")
fig.suptitle(
    f"DBSCAN sweep on Z_sub (n={len(Z_sub):,}), min_samples={DB_MIN_SAMPLES}, ball_tree",
    y=1.02,
)
fig.tight_layout()
fig.savefig(FIG / "07b_dbscan_clusters_noise_vs_eps.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(db_df["eps"], db_df["davies_bouldin_nonnoise"], "^-", color="olive", ms=6, lw=1.2)
ax.set_xlabel("eps")
ax.set_ylabel("Davies–Bouldin (non-noise only)")
ax.set_title("DBSCAN internal quality vs eps (lower is better when defined)")
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(FIG / "07c_dbscan_davies_bouldin_vs_eps.png", bbox_inches="tight")
plt.show()

heat = None
title = ""
if DATA_CLUSTER.exists():
    dc = pd.read_csv(DATA_CLUSTER).drop(columns=["Unnamed: 0"], errors="ignore")
    need = {"cluster", "month_of_year", "invalid_ratio"}
    if need.issubset(dc.columns):
        dc2 = dc.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna(
            subset=list(need)
        )
        if len(dc2) > 0:
            heat = dc2.pivot_table(
                index="cluster", columns="month_of_year", values="invalid_ratio", aggfunc="mean"
            )
            title = "Mean invalid_ratio: cluster × month (from with_cluster_sampled.csv)"

if heat is None or np.size(heat) == 0:
    kmap = 6
    lab_full = KMeans(n_clusters=kmap, random_state=RANDOM_STATE, n_init=10).fit_predict(Z)
    tmp = X_all.copy()
    tmp["cluster"] = lab_full
    tmp["invalid_ratio"] = y_all.values
    heat = tmp.pivot_table(index="cluster", columns="month_of_year", values="invalid_ratio", aggfunc="mean")
    title = f"Mean invalid_ratio: cluster × month (fallback KMeans k={kmap} on ST)"

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(heat, ax=ax, cmap="mako", annot=False, cbar_kws={"label": "mean invalid_ratio"})
ax.set_title(title)
fig.tight_layout()
fig.savefig(FIG / "08_cluster_month_heatmap.png", bbox_inches="tight")
plt.show()